# Create the DistilBERT LibTorch Artifact

Run this notebook inside the Triton Control code-server workspace. It exports `distilbert-base-uncased-finetuned-sst-2-english` as `distilbert_sentiment/1/model.pt` for Triton's PyTorch/LibTorch backend. This export requires a GPU.

In [ ]:
%pip install torch transformers

In [ ]:
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer


MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"
MAX_LENGTH = 32
OUTPUT_PATH = Path("distilbert_sentiment/1/model.pt")

In [ ]:
class SentimentWrapper(torch.nn.Module):
    def __init__(self, model: torch.nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.logits

In [ ]:
if not torch.cuda.is_available():
    raise SystemExit("CUDA is required to export this GPU LibTorch example.")

device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)
base_model.eval()

encoded = tokenizer(
    "This product is genuinely useful and easy to recommend.",
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

model = SentimentWrapper(base_model).to(device)
model.eval()

with torch.no_grad():
    traced_model = torch.jit.trace(
        model,
        (
            encoded["input_ids"].to(device=device, dtype=torch.long),
            encoded["attention_mask"].to(device=device, dtype=torch.long),
        ),
        strict=False,
    )

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
traced_model.save(str(OUTPUT_PATH))
print(f"Saved {OUTPUT_PATH}")